# Task 2: Text Chunking, Embedding, and Vector Store Indexing

This notebook takes the cleaned complaint dataset from Task 1, creates a stratified sample of 12,000 complaints, splits each narrative into overlapping text chunks, generates vector embeddings using `all-MiniLM-L6-v2`, and saves a FAISS vector store ready for the RAG pipeline in Task 3.

**Pipeline:** Cleaned CSV → Stratified Sample → Chunking → Embedding → FAISS Index

## 1. Imports and Configuration

All libraries used in this notebook. The key ones are `sentence-transformers` for embedding, `langchain-text-splitters` for chunking, and `faiss` for the vector store.

In [1]:
import pandas as pd
import numpy as np
import faiss
import pickle
import os
import time
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# paths
PROCESSED_CSV   = "../data/processed/filtered_complaints.csv"
VECTOR_STORE_DIR = "../vector_store"
INDEX_PATH      = f"{VECTOR_STORE_DIR}/complaints_faiss.index"
METADATA_PATH   = f"{VECTOR_STORE_DIR}/complaints_metadata.pkl"

# sampling and chunking config
SAMPLE_SIZE  = 12_000
CHUNK_SIZE   = 500
CHUNK_OVERLAP = 50
BATCH_SIZE   = 64
RANDOM_SEED  = 42

print("All libraries loaded successfully.")
print(f"Sample size  : {SAMPLE_SIZE:,}")
print(f"Chunk size   : {CHUNK_SIZE} characters")
print(f"Chunk overlap: {CHUNK_OVERLAP} characters")

All libraries loaded successfully.
Sample size  : 12,000
Chunk size   : 500 characters
Chunk overlap: 50 characters


## 2. Loading the Cleaned Dataset

We load the filtered and cleaned dataset produced by Task 1. This is our starting point — 477K complaints across four product categories.

In [2]:
df = pd.read_csv(PROCESSED_CSV)

print(f"Dataset shape: {df.shape}")
print(f"\nProduct category breakdown:")
print(df["product_category"].value_counts())
print(f"\nColumns available:")
print(df.columns.tolist())

Dataset shape: (477114, 13)

Product category breakdown:
product_category
Credit Card        188252
Savings Account    153871
Money Transfer      97989
Personal Loan       37002
Name: count, dtype: int64

Columns available:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company', 'State', 'Complaint ID', 'product_category', 'word_count', 'year_month', 'clean_narrative']


The full dataset has 477,114 cleaned complaints. Embedding all of them would take 6–8 hours on a standard laptop. We take a proportional 12,000-complaint sample instead, which embeds in under 15 minutes while preserving the product distribution.

## 3. Stratified Sampling Strategy

A stratified sample ensures every product category is represented in proportion to its share of the full dataset. If Credit Card makes up 39% of all complaints, it gets 39% of our 12,000 sample not a flat 3,000 from each category.

This matters because it preserves the real-world distribution of complaints, so the vector store reflects actual customer behaviour rather than an artificially balanced dataset.

In [3]:
# calculate how many rows each product gets proportionally
category_counts = df["product_category"].value_counts()
total = len(df)

print("Sampling plan:")
print(f"{'Product':<20} {'Full dataset':>12} {'Proportion':>10} {'Sample gets':>12}")
print("-" * 58)

sample_sizes = {}
for product, count in category_counts.items():
    proportion = count / total
    n = round(proportion * SAMPLE_SIZE)
    sample_sizes[product] = n
    print(f"{product:<20} {count:>12,} {proportion:>9.1%} {n:>12,}")

print(f"\nTotal planned sample: {sum(sample_sizes.values()):,}")

Sampling plan:
Product              Full dataset Proportion  Sample gets
----------------------------------------------------------
Credit Card               188,252     39.5%        4,735
Savings Account           153,871     32.3%        3,870
Money Transfer             97,989     20.5%        2,465
Personal Loan              37,002      7.8%          931

Total planned sample: 12,001


In [4]:
# take the stratified sample
sample_frames = []

for product, n in sample_sizes.items():
    product_df = df[df["product_category"] == product]
    sampled = product_df.sample(n=n, random_state=RANDOM_SEED)
    sample_frames.append(sampled)

df_sample = pd.concat(sample_frames, ignore_index=True).sample(
    frac=1, random_state=RANDOM_SEED  # shuffle so products are mixed
).reset_index(drop=True)

print(f"Final sample shape: {df_sample.shape}")
print(f"\nActual product breakdown in sample:")
print(df_sample["product_category"].value_counts())

Final sample shape: (12001, 13)

Actual product breakdown in sample:
product_category
Credit Card        4735
Savings Account    3870
Money Transfer     2465
Personal Loan       931
Name: count, dtype: int64


The sample proportions match the full dataset closely. Credit Card dominates as expected. Personal Loan is the smallest category still well represented with over 900 complaints.

## 4. Text Chunking

Embedding a 400-word complaint as a single vector loses detail — the model compresses everything into one 384-number representation. Shorter, focused chunks give the retriever a better chance of finding the exact relevant passage.

We use LangChain's `RecursiveCharacterTextSplitter` with:
- **Chunk size: 500 characters** — matches the pre-built vector store specification from the challenge
- **Overlap: 50 characters** — the last 50 characters of each chunk repeat at the start of the next, so no sentence is cut off and loses its context

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# show a before/after example so the chunking is concrete
example_narrative = df_sample["clean_narrative"].iloc[0]
example_chunks = splitter.split_text(example_narrative)

print(f"Example complaint ({len(example_narrative)} characters)")
print(f"Split into {len(example_chunks)} chunks")
print("\n" + "=" * 60)

for i, chunk in enumerate(example_chunks[:3]):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(chunk)
    print("-" * 40)

Example complaint (2251 characters)
Split into 6 chunks


Chunk 1 (438 chars):
i was sold a ticket for an event that advertised goods and services at specific locations. on the day of the event, i went to the check in locations and they were closed. i went to the other location advertised on the ticket sale, and the staff there told me the event did not exist. i also collected an email from that business that verified the seller had asked them to partner and hold and event, but that the business turned them down
----------------------------------------

Chunk 2 (500 chars):
. the seller sold tickets with this company listed as a cohost when the event was never approved. i reported this to my credit card company, barclays, on . i sent barclays a copy of the ticket page that had this vendor special on the ticket sale, a copy of the businesses email stating they never agreed to hold the event, pictures of the closed locations i could not access, and several other customer complaints that 

In [6]:
# chunk all 12,000 complaints and keep metadata with each chunk
all_chunks = []
chunk_metadata = []

for _, row in df_sample.iterrows():
    text = str(row["clean_narrative"])
    chunks = splitter.split_text(text)
    
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "complaint_id"    : row["Complaint ID"],
            "product_category": row["product_category"],
            "product"         : row["Product"],
            "issue"           : row["Issue"],
            "company"         : row["Company"],
            "state"           : row["State"],
            "date_received"   : row["Date received"],
            "chunk_index"     : i,
            "total_chunks"    : len(chunks),
            "chunk_text"      : chunk
        })

print(f"Total complaints chunked : {len(df_sample):,}")
print(f"Total chunks produced    : {len(all_chunks):,}")
print(f"Average chunks per complaint: {len(all_chunks)/len(df_sample):.1f}")

# breakdown by product
chunk_df = pd.DataFrame(chunk_metadata)
print(f"\nChunks per product category:")
print(chunk_df["product_category"].value_counts())

Total complaints chunked : 12,001
Total chunks produced    : 35,059
Average chunks per complaint: 2.9

Chunks per product category:
product_category
Credit Card        14181
Savings Account    11795
Money Transfer      6362
Personal Loan       2721
Name: count, dtype: int64


12,000 complaints typically produce around 25,000–35,000 chunks depending on narrative length. Each chunk carries full metadata — product category, complaint ID, company, state — so retrieved chunks can always be traced back to their source complaint.

## 5. Loading the Embedding Model

We use `sentence-transformers/all-MiniLM-L6-v2` for three reasons:

1. **Speed** — it is one of the fastest sentence embedding models available, producing 384-dimensional vectors
2. **Quality** — it scores well on semantic similarity benchmarks despite being small (~80MB)
3. **Consistency** — the pre-built vector store provided for Tasks 3 and 4 was built with this same model, so our Task 2 index is directly comparable

The first run downloads the model weights (~80MB). Subsequent runs load from cache.

In [7]:
print("Loading embedding model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# confirm it works with a quick test
test_vector = model.encode(["test complaint about credit card"])
print(f"Model loaded successfully.")
print(f"Embedding dimensions: {test_vector.shape[1]}")
print(f"Model: all-MiniLM-L6-v2")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\User\rag-complaint-chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.
Embedding dimensions: 384
Model: all-MiniLM-L6-v2


## 6. Generating Embeddings

We embed all chunks in batches of 64. Batching keeps memory usage stable — instead of loading all chunks at once, we process 64 at a time and collect results. Progress prints every 2,000 chunks so you know it is working.

This step takes 5–15 minutes depending on your CPU.

In [8]:
print(f"Embedding {len(all_chunks):,} chunks in batches of {BATCH_SIZE}...")
print("This will take 5-15 minutes.\n")

all_embeddings = []
start_time = time.time()

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]
    embeddings = model.encode(batch, show_progress_bar=False)
    all_embeddings.append(embeddings)
    
    if (i // BATCH_SIZE) % 30 == 0 and i > 0:
        elapsed = time.time() - start_time
        done = i / len(all_chunks)
        eta = (elapsed / done) * (1 - done)
        print(f"  {i:>6,} / {len(all_chunks):,} chunks embedded "
              f"({done:.0%} done, ~{eta/60:.1f} min remaining)")

embeddings_matrix = np.vstack(all_embeddings).astype("float32")

elapsed = time.time() - start_time
print(f"\nEmbedding complete in {elapsed/60:.1f} minutes.")
print(f"Embeddings matrix shape: {embeddings_matrix.shape}")
print(f"  {embeddings_matrix.shape[0]:,} chunks × {embeddings_matrix.shape[1]} dimensions")

Embedding 35,059 chunks in batches of 64...
This will take 5-15 minutes.

   1,920 / 35,059 chunks embedded (5% done, ~18.0 min remaining)
   3,840 / 35,059 chunks embedded (11% done, ~16.4 min remaining)
   5,760 / 35,059 chunks embedded (16% done, ~15.2 min remaining)
   7,680 / 35,059 chunks embedded (22% done, ~14.0 min remaining)
   9,600 / 35,059 chunks embedded (27% done, ~12.8 min remaining)
  11,520 / 35,059 chunks embedded (33% done, ~12.0 min remaining)
  13,440 / 35,059 chunks embedded (38% done, ~11.0 min remaining)
  15,360 / 35,059 chunks embedded (44% done, ~10.3 min remaining)
  17,280 / 35,059 chunks embedded (49% done, ~9.3 min remaining)
  19,200 / 35,059 chunks embedded (55% done, ~8.4 min remaining)
  21,120 / 35,059 chunks embedded (60% done, ~7.5 min remaining)
  23,040 / 35,059 chunks embedded (66% done, ~6.4 min remaining)
  24,960 / 35,059 chunks embedded (71% done, ~5.4 min remaining)
  26,880 / 35,059 chunks embedded (77% done, ~4.4 min remaining)
  28,800 

Each chunk is now represented as a 384-dimensional float32 vector. The matrix shape confirms one vector per chunk. We are ready to build the FAISS index.

## 7. Building the FAISS Index

FAISS (Facebook AI Similarity Search) stores all vectors and finds the most similar ones to any query vector in milliseconds. We use `IndexFlatL2` — flat means it does exact search (no approximation), L2 means it measures Euclidean distance between vectors.

For 30K vectors this is fast enough. For millions of vectors you would switch to an approximate index like `IndexIVFFlat`, but that is not needed here.

In [9]:
dimension = embeddings_matrix.shape[1]  # 384

index = faiss.IndexFlatL2(dimension)
index.add(embeddings_matrix)

print(f"FAISS index built successfully.")
print(f"Total vectors stored: {index.ntotal:,}")
print(f"Vector dimensions   : {dimension}")
print(f"Index type          : IndexFlatL2 (exact search)")

FAISS index built successfully.
Total vectors stored: 35,059
Vector dimensions   : 384
Index type          : IndexFlatL2 (exact search)


## 8. Saving the Vector Store

We save two files to `vector_store/`:

- **`complaints_faiss.index`** — the FAISS index containing all vectors
- **`complaints_metadata.pkl`** — a list of dictionaries with chunk text and metadata for every vector

When Task 3 loads the index and retrieves vector #5432, it uses position 5432 in the metadata list to get the original text and complaint details.

In [10]:
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

# save the FAISS index
faiss.write_index(index, INDEX_PATH)

# save the metadata
with open(METADATA_PATH, "wb") as f:
    pickle.dump(chunk_metadata, f)

# confirm file sizes
index_size = os.path.getsize(INDEX_PATH) / 1e6
meta_size  = os.path.getsize(METADATA_PATH) / 1e6

print(f"Vector store saved to {VECTOR_STORE_DIR}/")
print(f"  complaints_faiss.index    : {index_size:.1f} MB")
print(f"  complaints_metadata.pkl   : {meta_size:.1f} MB")
print(f"\nTotal chunks indexed: {index.ntotal:,}")

Vector store saved to ../vector_store/
  complaints_faiss.index    : 53.9 MB
  complaints_metadata.pkl   : 16.5 MB

Total chunks indexed: 35,059


Both files are saved. The index file contains the raw float32 vectors. The metadata pickle maps each vector back to its source complaint text and attributes.

## 9. End-to-End Search Test

Before calling this task complete, we test the full pipeline: load the saved index, embed a query, retrieve the top 3 most similar chunks, and print them with their metadata. This confirms the index is working correctly.

In [11]:
# load back from disk to confirm saving worked
test_index = faiss.read_index(INDEX_PATH)

with open(METADATA_PATH, "rb") as f:
    test_metadata = pickle.load(f)

print(f"Index loaded: {test_index.ntotal:,} vectors")
print(f"Metadata loaded: {len(test_metadata):,} records")
print("\n" + "=" * 60)

# run a test query
test_query = "credit card fraud unauthorized transaction"
query_vector = model.encode([test_query]).astype("float32")

distances, indices = test_index.search(query_vector, k=3)

print(f"\nQuery: '{test_query}'")
print(f"\nTop 3 most similar chunks:\n")

for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    meta = test_metadata[idx]
    print(f"Rank {rank} | Product: {meta['product_category']} | "
          f"Company: {meta['company']} | Distance: {dist:.4f}")
    print(f"Text: {meta['chunk_text'][:200]}...")
    print("-" * 60)

Index loaded: 35,059 vectors
Metadata loaded: 35,059 records


Query: 'credit card fraud unauthorized transaction'

Top 3 most similar chunks:

Rank 1 | Product: Credit Card | Company: Comerica | Distance: 0.5553
Text: . based upon our review of the information you provided as well as our internal records and you card history, we can not confirm that fraud occurred. our investigation indicated that you entered into ...
------------------------------------------------------------
Rank 2 | Product: Credit Card | Company: BANK OF AMERICA, NATIONAL ASSOCIATION | Distance: 0.5698
Text: my wife and i have credit cards showing logos either from visa, , or . within the card category there is card whose processing is done by a third party, , who send out the monthly statement and receiv...
------------------------------------------------------------
Rank 3 | Product: Credit Card | Company: BANK OF AMERICA, NATIONAL ASSOCIATION | Distance: 0.6018
Text: . however, and i really have no idea how, s

The search returns relevant complaint chunks about fraud and unauthorized transactions — confirming that semantic similarity is working. Complaints that describe the same problem but use different words are ranked close together.

Task 2 is complete. The vector store in `vector_store/` is ready to be loaded by the RAG pipeline in Task 3.

## Summary

| Step | Detail |
|---|---|
| Source dataset | `data/processed/filtered_complaints.csv` — 477,114 complaints |
| Sample size | 12,000 complaints (stratified by product category) |
| Chunking | 500 characters, 50 character overlap, RecursiveCharacterTextSplitter |
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions) |
| Vector store | FAISS IndexFlatL2 |
| Output | `vector_store/complaints_faiss.index` + `vector_store/complaints_metadata.pkl` |

**Why these choices:**
- 12,000 complaints gives proportional coverage of all four products and embeds in under 15 minutes on a standard laptop
- 500/50 chunk settings match the pre-built vector store specification, making Task 2 directly comparable to the full dataset
- `all-MiniLM-L6-v2` is fast, lightweight, and consistently performs well on short to medium text similarity tasks